In [4]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import warnings

working_dir = r'E:\Projects\Diffusion'
os.chdir(working_dir)
warnings.filterwarnings('ignore')

## Correlation Analysis

In [62]:
# Load the data
data = pd.read_csv('./warehouse/processed/benchmark_data_log_ret.csv')
data['date'] = pd.to_datetime(data['date'])
macro_indicators = pd.read_parquet('./warehouse/raw/all_macro_data.parquet').reset_index(drop=False)
macro_indicators_map = pd.read_csv('./warehouse/raw/macro_identifier.csv')

macro_indicators = macro_indicators.rename(columns={'as_of': 'date'})
macro_indicators['date'] = pd.to_datetime(macro_indicators['date'])
macro_indicators = macro_indicators.loc[(macro_indicators['date'] >= data['date'].min()) & (macro_indicators['date'] <= data['date'].max())].reset_index(drop=True)
"""macro_indicators.set_index('date', inplace=True)
macro_daily = macro_indicators.resample('D').ffill().reset_index(drop=False)"""

"macro_indicators.set_index('date', inplace=True)\nmacro_daily = macro_indicators.resample('D').ffill().reset_index(drop=False)"

In [65]:
macro_daily = pd.DataFrame()

for _, row in tqdm(macro_indicators.iterrows(), total=macro_indicators.shape[0], desc="Processing rows"):
    row = row.T
    
    start_date = row['date'].replace(day=1)
    end_date = row['date']
    
    date_range = pd.date_range(start=start_date, end=end_date)
    daily_df = pd.DataFrame({'date': date_range})
    
    for col in macro_indicators.columns:
        if col != 'date':
            daily_df[col] = row[col]
    
    macro_daily = pd.concat([macro_daily, daily_df])

macro_daily = macro_daily.loc[macro_daily['date'].isin(data['date'])].reset_index(drop=True)

Processing rows: 100%|██████████| 260/260 [00:24<00:00, 10.72it/s]


In [69]:
# calculate the correlation between macro indicators and stock returns
corr_dict = {}
for col in macro_daily.columns:
    corr_list = []
    if col != 'date':
        for return_col in data.columns:
            if return_col != 'date':
                corr = data[return_col].corr(macro_daily[col])
                corr_list.append(corr)
        corr_dict[col] = corr_list

In [76]:
# rank mean correlation, max correlation, min correlation descending
mean_corr = {}
max_corr = {}
min_corr = {}
for key, value in corr_dict.items():
    mean_corr[key] = sum(value) / len(value)
    max_corr[key] = max(value)
    min_corr[key] = min(value)
df_corr = pd.DataFrame({'macro': list(mean_corr.keys()),
                        'mean_corr': list(mean_corr.values()),
                        'max_corr': list(max_corr.values()),
                        'min_corr': list(min_corr.values())})
df_corr = df_corr.sort_values(by='mean_corr', ascending=False).reset_index(drop=True)
df_corr = df_corr.dropna()
print('Top 10 positively correlated macro indicators:')
print(df_corr.head(10))
print('Top 10 negatively correlated macro indicators:')
print(df_corr.tail(10))

Top 10 positively correlated macro indicators:
  macro  mean_corr  max_corr  min_corr
0  D274   0.045351  0.098284  0.016318
1  D532   0.036642  0.115021 -0.039107
2  D327   0.035797  0.073861  0.004691
3  D390   0.035731  0.071791  0.020522
4  D594   0.034808  0.071906  0.006844
5  D158   0.034805  0.071288  0.009938
6  D391   0.034351  0.058100  0.014103
7  D238   0.033822  0.065417  0.005013
8  D674   0.033458  0.076236  0.008118
9  D396   0.033403  0.075632  0.014560
Top 10 negatively correlated macro indicators:
    macro  mean_corr  max_corr  min_corr
678  D257  -0.049766 -0.015063 -0.122544
679  D442  -0.050585 -0.014752 -0.078035
680  D269  -0.051469 -0.033848 -0.124204
681  D258  -0.052177 -0.028014 -0.105510
682  D084  -0.052493  0.043353 -0.147088
683  D441  -0.052973 -0.019072 -0.090587
684  D646  -0.053482 -0.018156 -0.128153
685  D428  -0.053982 -0.027466 -0.134760
686  D404  -0.057089 -0.017047 -0.110422
687  D018  -0.064623 -0.026047 -0.099390


## 3 Regime Preprocessing

In [17]:
df_regime = pd.read_csv('./warehouse/raw/macro_dif_regimes.csv').rename(columns={'Unnamed: 0':'date', 'regime':'regime_prob'})
df_regime['date'] = pd.to_datetime(df_regime['date'])
df_regime['regime_prob'] = df_regime['regime_prob'].apply(lambda x: eval(x.replace('  ', ' ').replace(' ', ',').replace(',,', ',')))
df_regime['regime'] = df_regime['regime_prob'].apply(lambda x: f'regime{x.index(max(x))+1}')

df_raw_macro = pd.read_csv('./warehouse/raw/reduce_dif_macro_data.csv').rename(columns={'as_of':'date'})
df_raw_macro['date'] = pd.to_datetime(df_raw_macro['date'])

df_regime = df_regime.merge(df_raw_macro, on='date', how='left')

In [18]:
df_regime

,date,regime_prob,regime,Macro_Signal_1,Macro_Signal_2,Macro_Signal_3,Macro_Signal_4
0,2001-01-31,"[1.0, 3.39439185e-19, 3.51227103e-71]",regime1,-0.933509,-1.395562,0.695339,0.531251
1,2001-02-28,"[1.0, 1.33214705e-80, 2.07619091e-66]",regime1,-0.157297,-1.638298,0.880579,1.274051
2,2001-03-31,"[1.0, 5.10664755e-35, 5.76646995e-39]",regime1,-1.531246,-1.503035,0.093705,0.934173
3,2001-04-30,"[1.0, 1.26225922e-35, 7.03301023e-110]",regime1,-0.248530,-2.060112,-0.049939,1.041300
4,2001-05-31,"[1.0, 1.37895308e-82, 6.75445682e-149]",regime1,-0.998259,-2.063958,-0.623936,1.372830
...,...,...,...,...,...,...,...
272,2023-09-30,"[0.00098534, 0.02506248, 0.97395218]",regime3,-0.140850,0.215741,1.763779,-0.584881
273,2023-10-31,"[0.0693553171, 0.930605569, 3.91136613e-05]",regime2,-0.189232,0.071377,2.293591,-0.547961
274,2023-11-30,"[0.02702523, 0.36385887, 0.6091159]",regime3,-0.669823,0.853063,2.028303,-0.598371
275,2023-12-31,"[0.00559878, 0.97970393, 0.01469729]",regime2,-1.487893,1.285679,1.717101,-0.950020


In [19]:
df_regime.to_csv('./warehouse/processed/macro_regime_3.csv', index=False)